# 18.2 Nonlinear variables

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Although AMPL variables are declared for nonlinear programs in the same way as for
linear programs, two features of variables — initial values and automatic substitution —
are particularly useful in working with nonlinear models.
**Initial values of variables**

You may specify values for AMPL variables. Prior to optimization, these "initial"
values can be displayed and manipulated by AMPL commands. When you type solve,
they are passed to the solver, which may use them as a starting guess at the solution.
After the solver has finished its work, the initial values are replaced by the computed
optimal ones.

All of the AMPL features for assigning values to parameters are also available for
variables. A `var` declaration may also specify initial values in an optional := phrase; for
the transportation example, you can write

```ampl
var Ship {LINKS} >= 0, := 1;
```

to set every Ship[i,j] initially to 1, or

```ampl
var Ship {(i,j) in LINKS} >= 0, := cap[i,j] - 1;
```

to initialize each Ship[i,j] to 1 less than cap[i,j]. Alternatively, initial values
may be given in a data statement along with the other data for a model:

```ampl
var Ship:  FRA   DET   LAN   WIN    STL    FRE    LAF :=
     GARY  800   400   400   200    400    200    200
     CLEV  800   800   800   600    600    500    600
     PITT  800   800   800   200    300    800    500 ;
```

Any of the data statements for parameters can also be used for variables, as explained in
Section 9.4.

All of these features for assigning values to the regular ("primal") variables also
apply to the dual variables associated with constraints ([Section 12.5](../12/12_5_displaying_solution_values.ipynb)). AMPL interprets an
assignment to a constraint name as an assignment to the associated dual variable or (in
the terminology more common in nonlinear programming) to the associated Lagrange
multiplier. A few solvers, such as MINOS, can make use of initial values for these multipliers.

You can often speed up the work of the solver by suggesting good initial values. This
can be so even for linear programs, but the effect is much stronger in the nonlinear case.
The choice of an initial guess may determine what value of the objective is found to be
"optimal" by the solver, or even whether the solver finds any optimal solution at all.
These possibilities are discussed further in the last section of this chapter.

If you don't give any initial value for a variable, then AMPL will tentatively set it to
zero. If the solver incorporates a routine for determining initial values, then it may re-set
the values of any uninitialized variables, while making use of the values of variables that
have been initialized. Otherwise, uninitialized variables will be left at zero. Although
zero is an obvious starting point, it has no special significance; for some of the examples
that we will give in [Section 18.4](./18_4_pitfalls_of_nonlinear_programming.ipynb), the solver cannot optimize successfully unless the initial
values are `reset` away from zero.
**Automatic substitution of variables**

The issue of substituting variables has already arisen in an example of the previous
section, where we declared variables to represent the shipping costs, and then defined
them in terms of other variables by use of a constraint:

```ampl
subject to Cost_Relation {(i,j) in LINKS}:
       Cost[i,j] =
       (cost1[i,j] + cost2[i,j]*Ship[i,j]) / (1 + Ship[i,j]);
```

If the expression to the right of the = sign is substituted for every appearance of
Cost[i,j], the Cost variables can be eliminated from the model, and these constraints
need not be passed to the solver. There are two ways in which you can tell AMPL
to make such substitutions automatically.

First, by changing option substout from its default value of zero to one, you can
tell AMPL to look for all "defining" constraints that have the form shown above: a single
variable to the left of an = sign. When this alternative is employed, AMPL tries to use
as many of these constraints as possible to substitute variables out of the model. After
you have typed solve and a nonlinear program has been generated from a model and
data, the constraints are scanned in the order that they appeared in the model. A constraint
is identified as "defining" provided that
- it has just one variable to the left of an = sign;
- the left-hand variable's declaration did not specify any restrictions, such as integrality
or bounds; and
- the left-hand variable has not already appeared in a constraint identified as defining.

The expression to the right of the = sign is then substituted for every appearance of the
left-hand variable in the other constraints, and the defining constraint is dropped. These
rules give AMPL an easy way to avoid circular substitutions, but they do imply that the
nature and number of substitutions may depend on the ordering of the constraints.

As an alternative, if you want to specify explicitly that a certain collection of variables
is to be substituted out, use an = phrase in the declarations of the variables. For the preceding
example, you could write:

```ampl
var Cost {(i,j) in LINKS}
       = (cost1[i,j] + cost2[i,j]*Ship[i,j]) / (1 + Ship[i,j]);
```

Then the variables Cost[i,j] would be replaced everywhere by the expression following
the = sign. Declarations of this kind can appear in any order, `subject to` the usual
requirement that every variable appearing in an = phrase must be previously defined.

Variables that can be substituted out are not mathematically necessary to the optimization
problem. Nevertheless, they often serve an essential descriptive purpose; by
associating names with nonlinear expressions, they permit more complicated expressions
to be written understandably. Moreover, even though these variables have been removed
from the problem sent to the solver, their names remain available for use in browsing
through the results of optimization.
When the same nonlinear expression appears more than once in the objective and constraints,
introducing a defined variable to represent it may make the model more concise
as well as more readable. AMPL also processes such a substitution efficiently. In generating
a representation of the nonlinear program for the solver, AMPL does not substitute a
copy of the whole defining expression for each occurrence of a defined variable. Instead
it breaks the expression into a linear and a nonlinear part, and saves one copy of the nonlinear
part together with a list of the locations where its value is to be substituted; only the
terms of the linear part are substituted explicitly in multiple locations. This separate
treatment of linear terms is advantageous for solvers (such as MINOS) that handle the linear
terms specially, but it may be turned off by setting option linelim to zero.

From the solver's standpoint, substitutions reduce the number of constraints and variables,
but tend to make the constraint and objective expressions more complex. As a
result, there are circumstances in which a solver will perform better if defined variables
are not substituted out. When developing a new model, you may have to experiment to
determine which substitutions give the best results.